# .dpairs2 Oracle — Data Structure Guide
This notebook explains what's inside a .dpairs2 file,
how the data is organized, and what each piece means.
Goal: help you think about how to convert this into model training data.

## 1. Big Picture: What is a .dpairs2 file?

A DCFR solver builds a **game tree** starting from the flop. When the flop action ends and the turn card is about to be dealt, we hit a **boundary node** (a chance node). Instead of continuing into the turn/river subtree, the solver records the **counterfactual values (CFVs)** at that boundary.

A `.dpairs2` file stores these boundary CFVs for **every iteration** of the solver. This is the "perfect oracle" — if a neural network can learn to predict these CFVs, we can skip the turn/river subtree entirely.

## 2. Binary Layout

Let's read the header to see the concrete dimensions.

In [45]:
import struct
import numpy as np
import pandas as pd

path = "../data/oracles/9s6d6c.dpairs2"

with open(path, "rb") as f:
    raw = f.read()

print(f"Total file size: {len(raw):,} bytes ({len(raw)/1024/1024:.1f} MB)")
print()

# ── Header: 32 bytes ──
# Bytes   Field             Type
# 0-7     Magic             "DPAIRS2\0" (8 bytes)
# 8-11    Version           u32 LE
# 12-15   NumOOP            u32 LE    (number of OOP hole-card combos)
# 16-19   NumIP             u32 LE    (number of IP hole-card combos)
# 20-23   NumBoundaries     u32 LE    (number of turn boundary nodes in DFS order)
# 24-27   NumIterations     u32 LE    (DCFR iterations recorded)
# 28-31   StartingPot       f32 LE    (initial pot size in chips)

magic = raw[0:8]
version, num_oop, num_ip, num_boundaries, num_iterations = struct.unpack_from("<5I", raw, 8)
starting_pot = struct.unpack_from("<f", raw, 28)[0]

header_df = pd.DataFrame([
    ["Magic",         f"{magic}",     "8 bytes",  "File identifier"],
    ["Version",       version,        "u32 LE",   "Format version"],
    ["NumOOP",        num_oop,        "u32 LE",   "OOP hole-card combinations"],
    ["NumIP",         num_ip,         "u32 LE",   "IP hole-card combinations"],
    ["NumBoundaries", num_boundaries, "u32 LE",   "Turn boundary nodes (DFS order)"],
    ["NumIterations", num_iterations, "u32 LE",   "DCFR iterations recorded"],
    ["StartingPot",   starting_pot,   "f32 LE",   "Initial pot size (chips)"],
], columns=["Field", "Value", "Type", "Description"])

print("=== HEADER (32 bytes) ===")
header_df

Total file size: 25,004,192 bytes (23.8 MB)

=== HEADER (32 bytes) ===


,Field,Value,Type,Description
0,Magic,b'DPAIRS2\x00',8 bytes,File identifier
1,Version,2,u32 LE,Format version
2,NumOOP,863,u32 LE,OOP hole-card combinations
3,NumIP,526,u32 LE,IP hole-card combinations
4,NumBoundaries,25,u32 LE,Turn boundary nodes (DFS order)
5,NumIterations,180,u32 LE,DCFR iterations recorded
6,StartingPot,55.0,f32 LE,Initial pot size (chips)


## 3. What are "Boundaries"?

A **boundary** = a point in the flop game tree where the action ends and the turn card would be dealt.

Different flop action sequences create different boundaries:

```
                     ROOT (pot=55, OOP to act)
                    /           |            \
               Check          Bet 33        AllIn
              /    \            |              |
         IP Check  IP Bet   IP to act      TERMINAL
            |       ...    /    |     \
         CHANCE          Fold  Call  Raise50  ...
      [Boundary 0]        |     |       \
       pot = 55        TERM  CHANCE    IP to act
                          [Boundary 1]   /    \
                           pot = 121   Call  Raise100
                                        |       \
                                     CHANCE    ...
                                  [Boundary 2]
                                   pot = 188
```

Each boundary represents a **different pot size** reached through different betting lines.
The 25 boundaries in this file come from all possible flop action sequences that reach the turn.

**Boundaries are numbered in DFS order** (depth-first traversal of the game tree).
This means Boundary 0 is the first turn chance node encountered when traversing left-to-right.

## 4. What is a CFV vector?

At each boundary, for each player, the solver computes a **Counterfactual Value (CFV)** vector.

- **OOP's CFV vector** has `num_oop` (863) floats — one value per possible OOP hand
- **IP's CFV vector** has `num_ip` (526) floats — one value per possible IP hand

Each float answers: *"If I hold this specific hand, what is my expected value at this boundary, weighted by my probability of reaching here?"*

```
Boundary 3, OOP CFV vector (863 floats):
  hand[0] = -0.0415   ← "holding hand #0, OOP expects to lose 0.0415 chips here"
  hand[1] =  0.0280   ← "holding hand #1, OOP expects to gain 0.0280 chips here"
  hand[2] = -0.0003   ← "holding hand #2, roughly break-even"
  ...
  hand[862] = 0.0450
```

**Why different sizes?** OOP and IP have different ranges (sets of hands they can hold).
On board 9s6d6c, OOP opens wider (863 combos) while IP defends tighter (526 combos).

## 5. Per-Iteration Record Layout

After the 32-byte header, the file contains `num_iterations` records back-to-back.
Each record has a fixed structure:

In [46]:
# Size calculation for one iteration record
cfv_bytes_per_boundary = (num_oop + num_ip) * 4  # both players, 4 bytes per f32
metadata_bytes = 4 + 4 + 4                        # iteration(u32) + exploitability(f32) + conv_mode(u32)
bytes_per_iteration = metadata_bytes + num_boundaries * cfv_bytes_per_boundary

print("=== ONE ITERATION RECORD ===")
print()
print("  ┌─ Metadata (12 bytes) ─────────────────────────────┐")
print("  │  iteration:       u32 LE  (4 bytes)  — iter number │")
print("  │  exploitability:  f32 LE  (4 bytes)  — current gap │")
print("  │  convergence:     u32 LE  (4 bytes)  — 0 or 1      │")
print("  └────────────────────────────────────────────────────┘")
print()
print(f"  Then {num_boundaries} boundaries × 2 players:")
print()
for b in range(min(3, num_boundaries)):
    print(f"    Boundary {b}, OOP: {num_oop} × f32 = {num_oop*4:,} bytes")
    print(f"    Boundary {b}, IP:  {num_ip} × f32 = {num_ip*4:,} bytes")
if num_boundaries > 3:
    print(f"    ... ({num_boundaries - 3} more boundaries)")
    print(f"    Boundary {num_boundaries-1}, OOP: {num_oop} × f32 = {num_oop*4:,} bytes")
    print(f"    Boundary {num_boundaries-1}, IP:  {num_ip} × f32 = {num_ip*4:,} bytes")
print()
print(f"  CFV data per boundary:  ({num_oop} + {num_ip}) × 4 = {cfv_bytes_per_boundary:,} bytes")
print(f"  Total per iteration:    12 + {num_boundaries} × {cfv_bytes_per_boundary:,} = {bytes_per_iteration:,} bytes")
print(f"  Total data section:     {num_iterations} × {bytes_per_iteration:,} = {num_iterations * bytes_per_iteration:,} bytes")
print(f"  Total file:             32 (header) + {num_iterations * bytes_per_iteration:,} (data) = {32 + num_iterations * bytes_per_iteration:,} bytes")
print()
print(f"  Verify: actual file size = {len(raw):,} bytes  ✓" if len(raw) == 32 + num_iterations * bytes_per_iteration else f"  MISMATCH: actual = {len(raw):,}")

=== ONE ITERATION RECORD ===

  ┌─ Metadata (12 bytes) ─────────────────────────────┐
  │  iteration:       u32 LE  (4 bytes)  — iter number │
  │  exploitability:  f32 LE  (4 bytes)  — current gap │
  │  convergence:     u32 LE  (4 bytes)  — 0 or 1      │
  └────────────────────────────────────────────────────┘

  Then 25 boundaries × 2 players:

    Boundary 0, OOP: 863 × f32 = 3,452 bytes
    Boundary 0, IP:  526 × f32 = 2,104 bytes
    Boundary 1, OOP: 863 × f32 = 3,452 bytes
    Boundary 1, IP:  526 × f32 = 2,104 bytes
    Boundary 2, OOP: 863 × f32 = 3,452 bytes
    Boundary 2, IP:  526 × f32 = 2,104 bytes
    ... (22 more boundaries)
    Boundary 24, OOP: 863 × f32 = 3,452 bytes
    Boundary 24, IP:  526 × f32 = 2,104 bytes

  CFV data per boundary:  (863 + 526) × 4 = 5,556 bytes
  Total per iteration:    12 + 25 × 5,556 = 138,912 bytes
  Total data section:     180 × 138,912 = 25,004,160 bytes
  Total file:             32 (header) + 25,004,160 (data) = 25,004,192 bytes

  Verif

## 6. Parse the Full File

Now let's load everything into a structured format so we can inspect it.

In [47]:
def load_dpairs2(path):
    """Parse a .dpairs2 file into a structured dict."""
    with open(path, "rb") as f:
        data = f.read()

    # Header
    assert data[0:8] == b"DPAIRS2\x00", f"Bad magic: {data[0:8]}"
    version, n_oop, n_ip, n_bound, n_iter = struct.unpack_from("<5I", data, 8)
    pot = struct.unpack_from("<f", data, 28)[0]

    # Parse iterations
    offset = 32
    iterations = []
    for _ in range(n_iter):
        it_num, exploit = struct.unpack_from("<If", data, offset)
        conv = struct.unpack_from("<I", data, offset + 8)[0] != 0
        offset += 12

        # CFVs: boundary-major, player-minor
        # Layout: [B0_OOP, B0_IP, B1_OOP, B1_IP, ..., BN_OOP, BN_IP]
        cfvs = np.empty((n_bound, 2), dtype=object)
        for b in range(n_bound):
            for p in range(2):
                n = n_oop if p == 0 else n_ip
                cfvs[b, p] = np.frombuffer(data, dtype=np.float32, count=n, offset=offset).copy()
                offset += n * 4

        iterations.append({
            "iter": it_num,
            "exploitability": exploit,
            "convergence_mode": conv,
            "cfvs": cfvs,  # shape: (n_bound, 2), each entry is np.array of f32
        })

    return {
        "version": version,
        "num_oop": n_oop,
        "num_ip": n_ip,
        "num_boundaries": n_bound,
        "num_iterations": n_iter,
        "starting_pot": pot,
        "iterations": iterations,
    }

oracle = load_dpairs2(path)
print(f"Loaded: {oracle['num_iterations']} iterations × {oracle['num_boundaries']} boundaries")
print(f"Each boundary has: OOP vector[{oracle['num_oop']}] + IP vector[{oracle['num_ip']}]")

Loaded: 180 iterations × 25 boundaries
Each boundary has: OOP vector[863] + IP vector[526]


## 7. The Data as a 3D Tensor

Think of the entire file as **two 3D arrays** (one per player):

```
OOP tensor:  shape = (num_iterations, num_boundaries, num_oop)
IP tensor:   shape = (num_iterations, num_boundaries, num_ip)
```

Each "slice" along axis 0 is one iteration. Each "row" within that slice is one boundary.

In [48]:
# Build the 3D tensors
iters = oracle["iterations"]
n_iter = oracle["num_iterations"]
n_bound = oracle["num_boundaries"]
n_oop = oracle["num_oop"]
n_ip = oracle["num_ip"]

oop_tensor = np.zeros((n_iter, n_bound, n_oop), dtype=np.float32)
ip_tensor  = np.zeros((n_iter, n_bound, n_ip),  dtype=np.float32)

for t, it in enumerate(iters):
    for b in range(n_bound):
        oop_tensor[t, b, :] = it["cfvs"][b, 0]
        ip_tensor[t, b, :]  = it["cfvs"][b, 1]

print(f"OOP tensor shape: {oop_tensor.shape}  — (iterations, boundaries, hands)")
print(f"IP  tensor shape: {ip_tensor.shape}  — (iterations, boundaries, hands)")
print()
print(f"Total floats stored: {oop_tensor.size + ip_tensor.size:,}")
print(f"  OOP: {n_iter} × {n_bound} × {n_oop} = {oop_tensor.size:,}")
print(f"  IP:  {n_iter} × {n_bound} × {n_ip} = {ip_tensor.size:,}")

OOP tensor shape: (180, 25, 863)  — (iterations, boundaries, hands)
IP  tensor shape: (180, 25, 526)  — (iterations, boundaries, hands)

Total floats stored: 6,250,500
  OOP: 180 × 25 × 863 = 3,883,500
  IP:  180 × 25 × 526 = 2,367,000


## 8. Convergence Mode — The DCFR Regime Switch

The solver has two regimes that affect how it discounts regrets:

| | Normal Mode | Convergence Mode |
|---|---|---|
| **When** | exploit >= 1% pot | exploit < 1% pot |
| **alpha** | `t^1.5 / (t^1.5 + 1)` | `max(above, 0.9)` |
| **beta** | `0.5` | `0.9` |
| **gamma** | `t / (t + 1)` | `max(above, 0.9)` |
| **Effect** | Aggressive exploration | Stable fine-tuning |

The convergence_mode flag is stored **per iteration** because replay must use the exact same discount parameters. The flag is recorded at the time of each iteration, not recomputed.

In [49]:
rows = []
for it in iters:
    pct = it["exploitability"] / oracle["starting_pot"] * 100
    rows.append({
        "iter": it["iter"],
        "exploitability": round(it["exploitability"], 4),
        "% of pot": round(pct, 2),
        "convergence_mode": it["convergence_mode"],
    })

iter_df = pd.DataFrame(rows)

# Show key rows: first, last, every 10th, and the mode-switch iteration
switch_iter = iter_df.loc[iter_df["convergence_mode"].diff().fillna(0) != 0].index.tolist()
show_idx = sorted(set(
    [0, len(iter_df) - 1]
    + list(range(0, len(iter_df), 10))
    + switch_iter
))
iter_df.iloc[show_idx]

,iter,exploitability,% of pot,convergence_mode
0,0,48.0713,87.40,False
10,10,22.3624,40.66,False
20,20,14.7415,26.80,False
30,30,4.5800,8.33,False
40,40,2.6926,4.90,False
50,50,1.6671,3.03,False
60,60,1.2100,2.20,False
70,70,1.9307,3.51,False
80,80,0.9480,1.72,False
90,90,0.6602,1.20,False


## 9. Inspecting a Single Boundary's CFVs

Let's look at one specific boundary to understand the data concretely.
Each boundary has two CFV vectors per iteration — one for OOP, one for IP.

In [50]:
# Pick boundary 3 — a mid-tree boundary with interesting OOP/IP dynamics
b = 3

oop_cfv = oop_tensor[-1, b, :]  # last iteration
ip_cfv  = ip_tensor[-1, b, :]

print(f"=== Boundary {b} at final iteration (iter {n_iter-1}) ===\n")

summary = pd.DataFrame({
    "player": ["OOP", "IP"],
    "vector_len": [n_oop, n_ip],
    "min": [oop_cfv.min(), ip_cfv.min()],
    "max": [oop_cfv.max(), ip_cfv.max()],
    "mean": [oop_cfv.mean(), ip_cfv.mean()],
    "std": [oop_cfv.std(), ip_cfv.std()],
    "nonzero": [np.count_nonzero(oop_cfv), np.count_nonzero(ip_cfv)],
}).set_index("player")

print("Summary stats:")
display(summary)

# Show first 15 values side by side
print(f"\nFirst 15 values of each vector:")
preview = pd.DataFrame({
    f"OOP cfv[0..14]": oop_cfv[:15],
    f"IP cfv[0..14]": ip_cfv[:15],
})
preview.index.name = "hand_idx"
preview

=== Boundary 3 at final iteration (iter 179) ===

Summary stats:


,vector_len,min,max,mean,std,nonzero
player,,,,,,
OOP,863,-0.041448,0.044999,-0.018163,0.022693,863
IP,526,-0.091179,0.065640,-0.052850,0.035702,526



First 15 values of each vector:


,OOP cfv[0..14],IP cfv[0..14]
hand_idx,,
0,-0.037808,-0.082467
1,-0.037808,-0.082398
2,-0.036608,-0.083271
3,-0.040115,-0.082467
4,-0.040143,-0.082398
5,-0.040114,-0.083271
6,-0.036414,-0.081782
7,-0.035699,-0.081782
8,-0.005379,-0.082586


## 10. Zero vs Non-Zero Boundaries (Active vs Unreachable)

Some boundaries have all-zero CFVs for one player. This happens when a boundary is **unreachable** from that player's perspective (e.g., the opponent always folds before reaching it).

In [51]:
# Check which boundaries are active for each player at the final iteration
last = -1

rows = []
for b in range(n_bound):
    oop_active = bool(np.any(oop_tensor[last, b, :] != 0))
    ip_active  = bool(np.any(ip_tensor[last, b, :] != 0))
    rows.append({
        "boundary": b,
        "OOP active": "YES" if oop_active else "---",
        "IP active":  "YES" if ip_active  else "---",
        "OOP |mean|": np.abs(oop_tensor[last, b, :]).mean(),
        "IP |mean|":  np.abs(ip_tensor[last, b, :]).mean(),
    })

bound_df = pd.DataFrame(rows).set_index("boundary")

oop_active_count = (bound_df["OOP active"] == "YES").sum()
ip_active_count  = (bound_df["IP active"]  == "YES").sum()
both_active = ((bound_df["OOP active"] == "YES") & (bound_df["IP active"] == "YES")).sum()

print(f"OOP active: {oop_active_count}/{n_bound},  IP active: {ip_active_count}/{n_bound},  Both active: {both_active}/{n_bound}\n")
bound_df

OOP active: 14/25,  IP active: 22/25,  Both active: 11/25



,OOP active,IP active,OOP |mean|,IP |mean|
boundary,,,,
0,YES,YES,9.937041e-03,0.028537
1,YES,YES,2.908089e-02,0.010063
2,YES,YES,1.771074e-02,0.036064
3,YES,YES,2.697405e-02,0.060068
4,YES,---,3.349478e-03,0.000000
5,YES,---,4.869801e-02,0.000000
6,YES,---,5.145303e-02,0.000000
7,YES,YES,2.830477e-07,0.011822
8,---,YES,0.000000e+00,0.025021


## 11. How CFVs Evolve Across Iterations

The whole point of recording per-iteration CFVs is that the solver needs **the exact CFV from each iteration** during replay. Early iterations are noisy; later iterations converge.

This shows how the CFV for a single hand at a single boundary changes over time:

In [52]:
# Track a few specific hands across iterations for boundary 0, OOP
b = 0
hand_indices = [0, 100, 400, 800]

# Sample every 10th iteration + last
sample_iters = list(range(0, n_iter, 10)) + [n_iter - 1]
sample_iters = sorted(set(sample_iters))

traj_df = pd.DataFrame(
    {f"hand[{h}]": oop_tensor[sample_iters, b, h] for h in hand_indices},
    index=pd.Index(sample_iters, name="iter"),
)

print(f"Boundary {b}, OOP — CFV values converging over iterations:")
print("(noisy early → stable late)\n")
traj_df

Boundary 0, OOP — CFV values converging over iterations:
(noisy early → stable late)



,hand[0],hand[100],hand[400],hand[800]
iter,,,,
0,-0.009585,-0.009834,-0.006350,-0.002818
10,-0.014427,-0.013832,-0.010296,-0.012005
20,-0.008208,-0.008072,-0.005428,-0.006014
30,-0.007826,-0.007655,-0.005336,-0.006249
40,-0.008975,-0.008684,-0.006103,-0.007020
50,-0.007979,-0.007860,-0.005323,-0.006113
60,-0.007891,-0.007807,-0.005273,-0.005957
70,-0.007986,-0.007999,-0.005382,-0.006001
80,-0.008206,-0.008204,-0.005609,-0.006061


## 12. Summary: Key Dimensions for Modeling

When thinking about how to convert this to a neural network:

```
Inputs to predict boundary CFVs:
  - iteration number (t)           → affects DCFR discount weights
  - convergence_mode flag          → determines which discount regime
  - boundary index (b)             → which pot-size / action-sequence
  - current strategy state         → from the flop solver (cfreach, regrets)
  
Output the model must produce:
  - OOP CFV vector: float32[num_oop]
  - IP CFV vector:  float32[num_ip]
```

The model needs to replace the turn/river subtree solve. During DCFR iteration `t`, instead of recursing into the turn, the model predicts the CFV vector for each active boundary.

In [53]:
total_samples = n_iter * n_bound * 2

dim_df = pd.DataFrame([
    ["Iterations",       n_iter,  "varies per solve"],
    ["Boundaries",       n_bound, "varies per tree config"],
    ["OOP hands",        n_oop,   "varies per board + range"],
    ["IP hands",         n_ip,    "varies per board + range"],
    ["Starting pot",     oracle["starting_pot"], "chips"],
    ["Total (boundary, player) pairs per iter", n_bound * 2, "model calls per DCFR step"],
    ["Total samples in file", total_samples, f"{n_iter} × {n_bound} × 2"],
    ["OOP floats per sample", n_oop,  f"output vector size"],
    ["IP floats per sample",  n_ip,   f"output vector size"],
], columns=["Dimension", "Value", "Note"])

dim_df

,Dimension,Value,Note
0,Iterations,180.0,varies per solve
1,Boundaries,25.0,varies per tree config
2,OOP hands,863.0,varies per board + range
3,IP hands,526.0,varies per board + range
4,Starting pot,55.0,chips
5,"Total (boundary, player) pairs per iter",50.0,model calls per DCFR step
6,Total samples in file,9000.0,180 × 25 × 2
7,OOP floats per sample,863.0,output vector size
8,IP floats per sample,526.0,output vector size
